In [7]:
import trueq as tq
import trueq.simulation as tqs
import trueq.math as tqm
from trueq import Gate
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy.linalg import polar
from noise_models import *
################ DEFINE GATE SETS AND MATCHES #####################
non_paulis = [tq.Gate.s, tq.Gate.cliff8, tq.Gate.cliff10, tq.Gate.cliff11]
paulis = [tq.Gate.x, tq.Gate.y, tq.Gate.z, tq.Gate.id]
clifford_gates = [
    tq.Gate.cliff0, tq.Gate.cliff1, tq.Gate.cliff2, tq.Gate.cliff3,
    tq.Gate.cliff4, tq.Gate.cliff5, tq.Gate.cliff6, tq.Gate.cliff7,
    tq.Gate.cliff8, tq.Gate.cliff9, tq.Gate.cliff10, tq.Gate.cliff11,
    tq.Gate.cliff13, tq.Gate.cliff14, tq.Gate.cliff15,
    tq.Gate.cliff16, tq.Gate.cliff17, tq.Gate.cliff18, tq.Gate.cliff19,
    tq.Gate.cliff20, tq.Gate.cliff21, tq.Gate.cliff22, tq.Gate.cliff23
]

easy_gates = paulis + non_paulis
one_qubit_hard_gates = [tq.Gate.t, tq.Gate.h]
multi_qubit_hard_gates = [tq.Gate.cx]
hard_gates = multi_qubit_hard_gates + one_qubit_hard_gates

pauli_cycles = [tq.Cycle({0:pauli}) for pauli in paulis]
dihedral_cycles = [tq.Cycle({0:easy}) for easy in easy_gates]
two_pauli_cycles = [tq.Cycle({0:pauli1, 1:pauli2}) for pauli1 in paulis for pauli2 in paulis]
match_i = tqs.GateMatch(tq.Gate.i)
match_t = tqs.GateMatch(tq.Gate.t)
match_h = tqs.GateMatch(tq.Gate.h)
match_easy_gates = tqs.GateMatch(easy_gates)
match_cnot= tqs.GateMatch(Gate.cx)
match_clifford = tqs.GateMatch(clifford_gates)

################ PROCESS FIDELITY CALCULATION #####################

h_circs = [tq.Circuit([{0:easy},{0:tq.Gate.h}]) for easy in paulis]
t_circs = [tq.Circuit([{0:easy},{0:tq.Gate.t}]) for easy in paulis]
i_circs = [tq.Circuit([{0:easy}]) for easy in paulis]
cnot_circs = [tq.Circuit([{0:easy_1, 1:easy_2},{(0,1):tq.Gate.cx}]) for easy_1 in paulis for easy_2 in paulis]

circs = {Gate.h: h_circs, Gate.t: t_circs, Gate.i: i_circs, Gate.cx: cnot_circs}

def get_hard_cycles(circuit: tq.Circuit):

    return [circuit[2*k+1] for k in range(circuit.n_cycles//2 -1)]


def process_fidelity(circuit: tq.Circuit, simulator):
    ideal_unitary_matrix = tq.Simulator().operator(circuit= circuit).upgrade().mat()
    ideal_unitary_superop = tqm.Superop.from_rowstack(ideal_unitary_matrix) 
        
    noisy_unitary_matrix = simulator.operator(circuit= circuit).upgrade().mat()
    noisy_unitary_superop = tqm.Superop.from_rowstack(noisy_unitary_matrix) 


    proc_fid = (ideal_unitary_superop.adj @ noisy_unitary_superop).fidelity 
    return proc_fid


def hard_gates_fid(noisy_sim):
    dic = {}
    cnot_fid = sum([process_fidelity(circ, noisy_sim) for circ in cnot_circs]) / len(cnot_circs)
    t_fid = sum([process_fidelity(circ, noisy_sim) for circ in t_circs]) / len(t_circs)
    h_fid = sum([process_fidelity(circ, noisy_sim) for circ in h_circs]) / len(h_circs)
    i_fid = sum([process_fidelity(circ, noisy_sim) for circ in i_circs]) / len(i_circs)
    dic[tq.Gate.cx] = cnot_fid
    dic[tq.Gate.t] = t_fid
    dic[tq.Gate.h] = h_fid
    dic[tq.Gate.id] = i_fid
    return dic

################ GATE AVERAGING AND GAUGE OPTIMIZATION #####################

def average_gate_set(hard_gate, noisy_sim):

    def hard_cycle(hard_gate):
        if hard_gate == Gate.h:
            return tq.Cycle({0: Gate.h})
        if hard_gate == Gate.t:
            return tq.Cycle({0: Gate.t})
        if hard_gate == Gate.cx:
            return tq.Cycle({(0,1): Gate.cx})
        if hard_gate == Gate.i:
            return tq.Cycle({0: Gate.i})

        else:
            return None
    
    randomizing_cycles = pauli_cycles if hard_gate == Gate.h or hard_gate == Gate.i else pauli_cycles if hard_gate == Gate.t else two_pauli_cycles

    lst = []

    if hard_gate == Gate.i:
        for r_1 in randomizing_cycles:
            for r_2 in randomizing_cycles:
                for r_3 in randomizing_cycles:
                    for r_4 in randomizing_cycles:
                        circ = tq.Circuit([r_1, r_2, r_3, r_4])
                        ideal_PTM = tqm.Superop.from_rowstack(tq.Simulator().operator(circuit=circ).upgrade().mat()).ptm
                        noisy_PTM = tqm.Superop.from_rowstack(noisy_sim.operator(circuit=circ).upgrade().mat()).ptm
                        mat = ideal_PTM.conj().T @ noisy_PTM
                        lst.append(mat)
    
    else:

        for r_1 in randomizing_cycles:
            for r_2 in randomizing_cycles:
                for r_3 in randomizing_cycles:
                    for r_4 in randomizing_cycles:
                        circ = tq.Circuit([r_1, hard_cycle(hard_gate), r_2, hard_cycle(hard_gate), r_3, hard_cycle(hard_gate), r_4, hard_cycle(hard_gate)])
                        ideal_PTM = tqm.Superop.from_rowstack(tq.Simulator().operator(circuit=circ).upgrade().mat()).ptm
                        noisy_PTM = tqm.Superop.from_rowstack(noisy_sim.operator(circuit=circ).upgrade().mat()).ptm
                        mat = ideal_PTM.conj().T @ noisy_PTM
                        mat[np.abs(mat) < 1e-6] = 0
                        lst.append(mat)
    
    avg =  sum(lst) / len(lst)
    return avg

def eigenvector_closest_to_one(A):
    # Compute eigenvalues and eigenvectors
    vals, vecs = np.linalg.eig(A)

    # Find index of eigenvalue closest to 1
    idx = np.argmin(np.abs(vals - 1))

    # Corresponding eigenvalue & eigenvector
    eigenvalue = vals[idx]
    eigenvector = vecs[:, idx]

    return eigenvalue, eigenvector

def unvec(matrix: np.array):
    n = int(np.sqrt(np.shape(matrix)[0]))
    return matrix.T.reshape(n,n)

def unitary_gauge(hard_gate, noisy_sim):
    m = 4 if hard_gate == Gate.cx else 2
    avg = average_gate_set(hard_gate, noisy_sim)
    choi = tqm.Superop.from_ptm(avg).choi/m
    eigenval, eigenvec = eigenvector_closest_to_one(choi)
    print(eigenval)
    u, p = polar(unvec(eigenvec*np.sqrt(m)))
    return u

def gauge_unitaries(noisy_sim):
    dic = {}
    gauge_CNOT = unitary_gauge(Gate.cx,noisy_sim)
    print("CNOT gauge computed")
    gauge_H = unitary_gauge(Gate.h,noisy_sim)
    print("H gauge computed")
    gauge_I = unitary_gauge(Gate.i,noisy_sim)
    print("I gauge computed")
    gauge_T = unitary_gauge(Gate.t,noisy_sim)
    print("T gauge computed")
    dic[tq.Gate.cx] = gauge_CNOT
    dic[tq.Gate.h] = gauge_H
    dic[tq.Gate.id] = gauge_I
    dic[tq.Gate.t] = gauge_T
    return dic

def process_fidelity_gauge(simulator):
    dic = {}
    gauge = gauge_unitaries(simulator)
    h_gauge_ptm = tqm.Superop.from_unitary(gauge[tq.Gate.h]).ptm
    cnot_gauge_ptm = tqm.Superop.from_unitary(gauge[tq.Gate.cx]).ptm
    i_gauge_ptm = tqm.Superop.from_unitary(gauge[tq.Gate.id]).ptm
    t_gauge_ptm = tqm.Superop.from_unitary(gauge[tq.Gate.t]).ptm
    gauge_ptms = {tq.Gate.h: h_gauge_ptm, tq.Gate.t: t_gauge_ptm, tq.Gate.cx: cnot_gauge_ptm, tq.Gate.id: i_gauge_ptm}
    for gate in gauge_ptms.keys():
        fidelities = []
        for circuit in circs[gate]:
            ideal_unitary_matrix = tq.Simulator().operator(circuit=circuit).upgrade().mat()
            ideal_unitary_superop = tqm.Superop.from_rowstack(ideal_unitary_matrix) 
            ideal_unitary_ptm = ideal_unitary_superop.ptm
            ideal_gauged_ptm = gauge_ptms[gate].conj().T @ ideal_unitary_ptm @ gauge_ptms[gate]
            ideal_gauged_superop = tqm.Superop.from_ptm(ideal_gauged_ptm)
            noisy_unitary_matrix = simulator.operator(circuit=circuit).upgrade().mat()
            noisy_unitary_superop = tqm.Superop.from_rowstack(noisy_unitary_matrix) 
            proc_fid = (ideal_gauged_superop.adj @ noisy_unitary_superop).fidelity 
            fidelities.append(proc_fid)
        dic[gate] = sum(fidelities) / len(fidelities)
    return dic, gauge

def gauge_fidelity(gauge: dict):
    dic = {}
    for gate in gauge.keys():
        dic[gate] = tqm.Superop.from_unitary(gauge[gate]).fidelity
    return dic

In [8]:
import pandas as pd

# CSV file path
csv_file = 'test_t_gate.csv'

df = pd.DataFrame(columns=['Error_Model', 'Strength', 'Process_Infidelity', 'Gate', 'F_CB', 'F_SCG', 'Gauge_Fidelity', 'F_CB_hat'])

# Dictionary of all simulators with their names
all_simulators = {
    'Gate_Independent': sim_gate_ind,
    'Gate_Dependent': sim_gate_dep,
    'ZXZXZ': sim_ZXZXZ,
    'ZXZXZ_Tilted': sim_ZXZXZ_tilted
}

# Process each error model
for error_model_name, simulators_dict in all_simulators.items():
    print(f"\nProcessing {error_model_name}...")
    
    for strength_level, (noisy_sim, process_infid) in simulators_dict.items():
        print(f"  Strength {strength_level}...")
        
        f_cb_dict = hard_gates_fid(noisy_sim)
        f_scg_dict, gauge_dict = process_fidelity_gauge(noisy_sim)
        
        # Gauge fidelity (same function for both)
        gauge_fid_dict = gauge_fidelity(gauge_dict)
        
        # Create a row for each gate
        for gate in f_cb_dict.keys():
            row_data = {
                'Error_Model': error_model_name,
                'Strength': strength_level,
                'Process_Infidelity': process_infid,
                'Gate': str(gate),
                'F_CB': f_cb_dict[gate],
                'F_SCG': f_scg_dict[gate],
                'Gauge_Fidelity': gauge_fid_dict[gate],
                'F_CB_hat': None
            }
            
            # Append row to dataframe
            df = pd.concat([df, pd.DataFrame([row_data])], ignore_index=True)
            
            # Save to CSV immediately after each row
            df.to_csv(csv_file, index=False)
            print(f"    Saved row for gate {gate}")

print(f"\n✓ All data saved to {csv_file}")
print(f"Total rows: {len(df)}")
df.head(10)



Processing Gate_Independent...
  Strength 1...
(0.9568430615191907-6.7693375717681416e-21j)
CNOT gauge computed
(0.9943544776674775+2.1382117680737565e-50j)
H gauge computed
(0.9997499697681316-4.591774807899561e-41j)
I gauge computed
(0.9800530295715224+2.2569491535787916e-34j)
T gauge computed
    Saved row for gate Gate.cx
    Saved row for gate Gate.t
    Saved row for gate Gate.h
    Saved row for gate Gate.id
  Strength 2...


         (/var/folders/wz/lbx10qps1tdc1npmd3kbq2t00000gn/T/ipykernel_80916/341829458.py:43)


(0.9519737825568326-8.449252926692954e-21j)
CNOT gauge computed
(0.9922077309757195-9.629649721936184e-35j)
H gauge computed
(0.999000504129897-9.027796614315164e-36j)
I gauge computed
(0.979318149979907-4.001486544309729e-34j)
T gauge computed
    Saved row for gate Gate.cx
    Saved row for gate Gate.t
    Saved row for gate Gate.h
    Saved row for gate Gate.id
  Strength 3...
(0.9462036713233142-4.23224193718182e-21j)
CNOT gauge computed
(0.9895725110211027-4.814751392571163e-35j)
H gauge computed
(0.9977534761323433+0j)
I gauge computed
(0.9780949123260487-4.7572256377777954e-35j)
T gauge computed
    Saved row for gate Gate.cx
    Saved row for gate Gate.t
    Saved row for gate Gate.h
    Saved row for gate Gate.id
  Strength 4...
(0.9395545843678798-2.0334109246935256e-20j)
CNOT gauge computed
(0.9864534071978347-1.4444474582904267e-34j)
H gauge computed
(0.9960120004451085-2.407412430484045e-35j)
I gauge computed
(0.9763856543390448-2.5823452753405965e-35j)
T gauge computed
  

,Error_Model,Strength,Process_Infidelity,Gate,F_CB,F_SCG,Gauge_Fidelity,F_CB_hat
0,Gate_Independent,1,0.000063,Gate.cx,0.988970,0.988970,1.0,None
1,Gate_Independent,1,0.000063,Gate.t,0.994938,0.994938,1.0,None
2,Gate_Independent,1,0.000063,Gate.h,0.998584,0.998584,1.0,None
3,Gate_Independent,1,0.000063,Gate.id,0.999937,0.999937,1.0,None
4,Gate_Independent,2,0.000250,Gate.cx,0.987693,0.987693,1.0,None
5,Gate_Independent,2,0.000250,Gate.t,0.994751,0.994751,1.0,None
6,Gate_Independent,2,0.000250,Gate.h,0.998044,0.998044,1.0,None
7,Gate_Independent,2,0.000250,Gate.id,0.999750,0.999750,1.0,None
8,Gate_Independent,3,0.000563,Gate.cx,0.986170,0.986170,1.0,None
9,Gate_Independent,3,0.000563,Gate.t,0.994440,0.994440,1.0,None


In [10]:
import pandas as pd

# Read both CSVs
sim = pd.read_csv("simulation_results_combined copy.csv")
tgate = pd.read_csv("test_t_gate.csv")

# Columns to copy over from test_t_gate (everything except F_CB_hat)
cols_to_copy = [c for c in tgate.columns if c != "F_CB_hat"]

# Option 1: if you just want the test_t_gate data with F_CB_hat from simulation_results.csv
# (assuming they align row-by-row):
merged = tgate.copy()
if "F_CB_hat" in sim.columns:
    merged["F_CB_hat"] = sim["F_CB_hat"].values
else:
    merged["F_CB_hat"] = None  # or leave absent if you prefer

# Save to new CSV
output_file = "simulation_results_t_gate_replaced.csv"
merged.to_csv(output_file, index=False)
print(f"Saved merged data to {output_file}")

Saved merged data to simulation_results_t_gate_replaced.csv
